<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/27-10-2026_PySpark_Exersices/PySpark_Hands_on_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PySpark Hands-on Exercises**

In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
import os

spark = (
    SparkSession.builder
    .appName("PySparkPractice")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)

stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)

products_df.show(truncate=False)
stores_df.show(truncate=False)

+----------+------------+-----------+----------+
|product_id|product_name|category   |list_price|
+----------+------------+-----------+----------+
|P01       |Laptop      |Electronics|60000     |
|P02       |Monitor     |Electronics|14000     |
|P03       |Headphones  |Accessories|3500      |
|P04       |Printer     |Office     |17000     |
|P05       |Router      |Networking |4500      |
|P06       |Tablet      |Electronics|24000     |
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|city     |region|
+--------+---------+------+
|S01     |Hyderabad|South |
|S02     |Bengaluru|South |
|S03     |Mumbai   |West  |
|S04     |Delhi    |North |
+--------+---------+------+



In [4]:
sales_base_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(((F.col("order_id") % 6) + 1).cast("string"), 2, "0")
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5).cast("int")
        )
    )
)

In [5]:
sales_df = (
    sales_base_df
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )
    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )
    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(((F.col("order_id") % 12) + 1).cast("string"), 2, "0")
            )
        )
    )
    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )
    .select(
        "order_id", "order_date", "store_id", "product_id", "quantity",
        "unit_price", "discount_pct", "order_status", "payment_method",
        "salesperson_id"
    )
)

print("Total records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

Total records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |C

**Practice Exercises**

In [6]:
ex1_df = sales_df.select("order_id", "product_id", "quantity", "unit_price")
ex1_df.show(10, truncate=False)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|3       |P04       |4       |17000     |
|9       |P04       |2       |17000     |
|2       |P03       |3       |3500      |
|8       |P03       |1       |3500      |
|4       |P05       |5       |4500      |
|10      |P05       |3       |4500      |
|5       |P06       |6       |24000     |
|11      |P06       |4       |24000     |
|1       |P02       |2       |14000     |
|7       |P02       |8       |14000     |
+--------+----------+--------+----------+
only showing top 10 rows


In [7]:
ex2_df = sales_df.filter((F.col("order_status") == "Completed") & (F.col("quantity") > 5))
ex2_df.show(10, truncate=False)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|15      |2026-01-16|S02     |P04       |8       |17000     |15          |Completed   |UPI           |EMP04         |
|39      |2026-02-09|S02     |P04       |8       |17000     |15          |Completed   |UPI           |EMP04         |
|38      |2026-02-08|S01     |P03       |7       |3500      |NULL        |Completed   |Cash          |EMP03         |
|5       |2026-01-06|S02     |P06       |6       |24000     |5           |Completed   |Cash          |EMP06         |
|23      |2026-01-24|S04     |P06       |8       |24000     |0           |Completed   |Cash          |EMP12         |
|29      |2026-01-30|S02     |P06       |6       |24000 

In [13]:
ex3_df = sales_df.withColumn("gross_revenue",F.col("quantity") * F.col("unit_price"))
ex3_df.show(10, truncate=False)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |68000        |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |Completed   |UPI           |EMP10         |34000        |
|2       |2026-01-03|S01     |P03       |3       |3500      |0           |Completed   |Cash          |EMP03         |10500        |
|8       |2026-01-09|S03     |P03       |1       |3500      |5           |Completed   |Cash          |EMP09         |3500         |
|4       |2026-01-05|S02     |P05       |5       |4500      |0           |Co

In [14]:
ex4_df = (sales_df.withColumnRenamed("order_status", "transaction_status").drop("payment_method"))
ex4_df.show(10, truncate=False)

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed         |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |Completed         |EMP10         |
|2       |2026-01-03|S01     |P03       |3       |3500      |0           |Completed         |EMP03         |
|8       |2026-01-09|S03     |P03       |1       |3500      |5           |Completed         |EMP09         |
|4       |2026-01-05|S02     |P05       |5       |4500      |0           |Completed         |EMP05         |
|10      |2026-01-11|S04     |P05       |3       |4500      |10          |Completed         |EMP11         |
|5       |2026-01-0

In [10]:
ex5_df = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity_sold")).orderBy("product_id")
ex5_df.show()

+----------+-------------------+
|product_id|total_quantity_sold|
+----------+-------------------+
|       P01|              66668|
|       P02|              83336|
|       P03|              66667|
|       P04|              83334|
|       P05|              66665|
|       P06|              83330|
+----------+-------------------+



In [15]:
sales_products_df = sales_df.join(products_df, "product_id", "left")
ex6_df = (sales_products_df
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("category")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"))
    .orderBy(F.desc("total_gross_revenue")))
ex6_df.show(truncate=False)

+-----------+-------------------+
|category   |total_gross_revenue|
+-----------+-------------------+
|Electronics|7166704000         |
|Office     |1416678000         |
|Networking |299992500          |
|Accessories|233334500          |
+-----------+-------------------+



In [17]:
sales_stores_df = sales_df.join(stores_df, "store_id", "left")
ex7_df = sales_stores_df.groupBy("city").agg(F.count("*").alias("total_transactions")).orderBy(F.desc("total_transactions"))
ex7_df.show(truncate=False)

+---------+------------------+
|city     |total_transactions|
+---------+------------------+
|Bengaluru|25001             |
|Hyderabad|25001             |
|Mumbai   |24999             |
|Delhi    |24999             |
+---------+------------------+



In [18]:
ex8_df = (
    sales_df.filter(F.col("order_status") == "Completed")
    .join(products_df, "product_id", "left")
    .join(stores_df, "store_id", "left")
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("product_id", "product_name", "category")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"))
    .orderBy(F.desc("total_gross_revenue"))
    .limit(5))
ex8_df.show(truncate=False)

+----------+------------+-----------+-------------------+
|product_id|product_name|category   |total_gross_revenue|
+----------+------------+-----------+-------------------+
|P01       |Laptop      |Electronics|3116940000         |
|P06       |Tablet      |Electronics|1558416000         |
|P04       |Printer     |Office     |1103912000         |
|P02       |Monitor     |Electronics|909076000          |
|P05       |Router      |Networking |233766000          |
+----------+------------+-----------+-------------------+



In [19]:
ex9_df = sales_df.select(F.sum(F.col("discount_pct").isNull().cast("int")).alias("null_discount_count"), F.sum(F.col("salesperson_id").isNull().cast("int")).alias("null_salesperson_count"))
ex9_df.show()

+-------------------+----------------------+
|null_discount_count|null_salesperson_count|
+-------------------+----------------------+
|               5263|                  5882|
+-------------------+----------------------+



In [20]:
ex10_df = sales_df.na.fill({"discount_pct": 0})
ex10_df.select("order_id", "discount_pct").show(10)

+--------+------------+
|order_id|discount_pct|
+--------+------------+
|       3|           0|
|       9|           5|
|       2|           0|
|       8|           5|
|       4|           0|
|      10|          10|
|       5|           5|
|      11|          10|
|       1|           0|
|       7|           5|
+--------+------------+
only showing top 10 rows


In [23]:
ex11_df = sales_df.na.drop(subset=["salesperson_id"])
print("Rows before:", sales_df.count())
print("Rows after:", ex11_df.count())

Rows before: 100000
Rows after: 94118


In [24]:
ex12_df = sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity").between(1, 3), "Small")
     .when(F.col("quantity").between(4, 6), "Medium")
     .otherwise("Large"))
ex12_df.select("order_id", "quantity", "order_size").show(12)

+--------+--------+----------+
|order_id|quantity|order_size|
+--------+--------+----------+
|       3|       4|    Medium|
|       9|       2|     Small|
|       2|       3|     Small|
|       8|       1|     Small|
|       4|       5|    Medium|
|      10|       3|     Small|
|       5|       6|    Medium|
|      11|       4|    Medium|
|       1|       2|     Small|
|       7|       8|     Large|
|      13|       6|    Medium|
|       6|       7|     Large|
+--------+--------+----------+
only showing top 12 rows


In [26]:
product_revenue_df = (sales_df.join(products_df.select("product_id", "product_name", "category"), "product_id")
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("product_id", "product_name", "category")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue")))
category_window = Window.partitionBy("category").orderBy(F.desc("total_gross_revenue"))
ex13_df = product_revenue_df.withColumn("revenue_rank", F.dense_rank().over(category_window))
ex13_df.orderBy("category", "revenue_rank").show(truncate=False)

+----------+------------+-----------+-------------------+------------+
|product_id|product_name|category   |total_gross_revenue|revenue_rank|
+----------+------------+-----------+-------------------+------------+
|P03       |Headphones  |Accessories|233334500          |1           |
|P01       |Laptop      |Electronics|4000080000         |1           |
|P06       |Tablet      |Electronics|1999920000         |2           |
|P02       |Monitor     |Electronics|1166704000         |3           |
|P05       |Router      |Networking |299992500          |1           |
|P04       |Printer     |Office     |1416678000         |1           |
+----------+------------+-----------+-------------------+------------+



In [28]:
running_window = (Window.partitionBy("store_id")
    .orderBy("order_date", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow))
ex14_df = (sales_df
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("running_gross_revenue", F.sum("gross_revenue").over(running_window)))
ex14_df.select("store_id", "order_date", "order_id", "gross_revenue", "running_gross_revenue").orderBy("store_id", "order_date", "order_id").show(20, truncate=False)

+--------+----------+--------+-------------+---------------------+
|store_id|order_date|order_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+-------------+---------------------+
|S01     |2026-01-01|180     |300000       |300000               |
|S01     |2026-01-01|360     |60000        |360000               |
|S01     |2026-01-01|540     |300000       |660000               |
|S01     |2026-01-01|720     |60000        |720000               |
|S01     |2026-01-01|900     |300000       |1020000              |
|S01     |2026-01-01|1080    |60000        |1080000              |
|S01     |2026-01-01|1260    |300000       |1380000              |
|S01     |2026-01-01|1440    |60000        |1440000              |
|S01     |2026-01-01|1620    |300000       |1740000              |
|S01     |2026-01-01|1800    |60000        |1800000              |
|S01     |2026-01-01|1980    |300000       |2100000              |
|S01     |2026-01-01|2160    |60000        |2160000           

In [30]:
ex15_df = (sales_df.filter(F.col("order_status") == "Completed")
    .withColumn("month", F.date_format("order_date", "yyyy-MM"))
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("month")
    .agg(F.sum("gross_revenue").alias("monthly_sales_revenue"))
    .orderBy("month"))
ex15_df.show(truncate=False)

+-------+---------------------+
|month  |monthly_sales_revenue|
+-------+---------------------+
|2026-01|1278728500           |
|2026-02|1027585500           |
|2026-03|1212087500           |
|2026-04|1218921000           |
|2026-05|1277560500           |
|2026-06|1089027500           |
+-------+---------------------+



In [31]:
lazy_df = sales_df.filter(F.col("quantity") > 5)
print("Transformation 1 defined: filter(quantity > 5)")

lazy_df = lazy_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
print("Transformation 2 defined: withColumn(gross_revenue)")

lazy_df = lazy_df.select("order_id", "product_id", "quantity", "gross_revenue")
print("Transformation 3 defined: select(columns)")

lazy_df.show(10, truncate=False)

Transformation 1 defined: filter(quantity > 5)
Transformation 2 defined: withColumn(gross_revenue)
Transformation 3 defined: select(columns)
+--------+----------+--------+-------------+
|order_id|product_id|quantity|gross_revenue|
+--------+----------+--------+-------------+
|15      |P04       |8       |136000       |
|21      |P04       |6       |102000       |
|14      |P03       |7       |24500        |
|22      |P05       |7       |31500        |
|5       |P06       |6       |144000       |
|23      |P06       |8       |192000       |
|29      |P06       |6       |144000       |
|7       |P02       |8       |112000       |
|13      |P02       |6       |84000        |
|6       |P01       |7       |420000       |
+--------+----------+--------+-------------+
only showing top 10 rows


In [33]:
ex17_df = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity"))
ex17_df.explain(mode="formatted")
ex17_rows = ex17_df.collect()
print(ex17_rows)

== Physical Plan ==
AdaptiveSparkPlan (14)
+- HashAggregate (13)
   +- HashAggregate (12)
      +- Project (11)
         +- SortMergeJoin LeftOuter (10)
            :- Sort (4)
            :  +- Exchange (3)
            :     +- Project (2)
            :        +- Range (1)
            +- Sort (9)
               +- Exchange (8)
                  +- Project (7)
                     +- Filter (6)
                        +- Scan ExistingRDD (5)


(1) Range
Output [1]: [id#30L]
Arguments: Range (1, 100001, step=1, splits=Some(2))

(2) Project
Output [2]: [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
Input [1]: [id#30L]

(3) Exchange
Input [2]: [product_id#33, quantity#35]
Arguments: hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=7461]

(4) Sort
Input [2]: [product_id#33, quantity#35]
Arguments: [product_id#33 ASC NULLS FIRST], false, 0

(5) Scan ExistingRDD
Output [4]: [product_id#0, product

In [34]:
completed_cached_df = sales_df.filter(F.col("order_status") == "Completed").cache()

start = time.time()
count_first = completed_cached_df.count()
first_seconds = time.time() - start

start = time.time()
count_second = completed_cached_df.count()
second_seconds = time.time() - start

print(f"First count: {count_first:,}; time: {first_seconds:.3f} seconds")
print(f"Second count: {count_second:,}; time: {second_seconds:.3f} seconds")

completed_cached_df.unpersist()

First count: 77,923; time: 4.866 seconds
Second count: 77,923; time: 1.967 seconds


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

In [36]:
high_quantity_df = sales_df.filter(F.col("quantity") > 5).persist(StorageLevel.MEMORY_AND_DISK)

print("Persisted row count:", high_quantity_df.count())
high_quantity_df.groupBy("store_id").count().show()

high_quantity_df.unpersist()
print("Persisted DataFrame released.")

Persisted row count: 37500
+--------+-----+
|store_id|count|
+--------+-----+
|     S02| 8334|
|     S04|12498|
|     S01| 8334|
|     S03| 8334|
+--------+-----+

Persisted DataFrame released.


In [37]:
cached_agg_df = (sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("product_id")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"))
    .cache())

cached_agg_df.count()
cached_agg_df.explain(mode="formatted")
cached_agg_df.show(truncate=False)
cached_agg_df.unpersist()

== Physical Plan ==
AdaptiveSparkPlan (29)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (28)
               +- == Final Plan ==
                  ResultQueryStage (19)
                  +- * HashAggregate (18)
                     +- * HashAggregate (17)
                        +- * Project (16)
                           +- * BroadcastHashJoin LeftOuter BuildRight (15)
                              :- ShuffleQueryStage (6), Statistics(sizeInBytes=3.1 MiB, rowCount=1.00E+5)
                              :  +- Exchange (5)
                              :     +- * Project (4)
                              :        +- * Range (3)
                              +- BroadcastQueryStage (14), Statistics(sizeInBytes=8.0 MiB, rowCount=6)
                                 +- BroadcastExchange (13)
                                    +- AQEShuffleRead (12)
                                       +- ShuffleQueryStage (11), Statistics(sizeInBytes=192.0 B, row

DataFrame[product_id: string, total_gross_revenue: bigint]

In [39]:
print("Existing partition count:", sales_df.rdd.getNumPartitions())
sales_8_df = sales_df.repartition(8)
print("After repartition(8):", sales_8_df.rdd.getNumPartitions())

Existing partition count: 2
After repartition(8): 8


In [41]:
partition_counts_df = (sales_8_df
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id"))
partition_counts_df.show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [42]:
sales_by_store_partition_df = sales_df.repartition(6, "store_id")
store_partition_map_df = (sales_by_store_partition_df
    .select("store_id", F.spark_partition_id().alias("partition_id"))
    .distinct()
    .orderBy("partition_id", "store_id"))
store_partition_map_df.show(truncate=False)

+--------+------------+
|store_id|partition_id|
+--------+------------+
|S03     |1           |
|S02     |3           |
|S01     |4           |
|S04     |5           |
+--------+------------+



In [43]:
sales_12_df = sales_df.repartition(12)
sales_coalesce_4_df = sales_12_df.coalesce(4)
sales_repartition_4_df = sales_12_df.repartition(4)

print("12-partition DataFrame:", sales_12_df.rdd.getNumPartitions())
print("After coalesce(4):", sales_coalesce_4_df.rdd.getNumPartitions())
print("After repartition(4):", sales_repartition_4_df.rdd.getNumPartitions())

print("\n--- coalesce(4) execution plan ---")
sales_coalesce_4_df.explain(mode="formatted")

print("\n--- repartition(4) execution plan ---")
sales_repartition_4_df.explain(mode="formatted")

12-partition DataFrame: 12
After coalesce(4): 4
After repartition(4): 4

--- coalesce(4) execution plan ---
== Physical Plan ==
AdaptiveSparkPlan (28)
+- == Final Plan ==
   ResultQueryStage (19)
   +- Coalesce (18)
      +- ShuffleQueryStage (17), Statistics(sizeInBytes=12.8 MiB, rowCount=1.00E+5)
         +- Exchange (16)
            +- * Project (15)
               +- * BroadcastHashJoin LeftOuter BuildRight (14)
                  :- AQEShuffleRead (5)
                  :  +- ShuffleQueryStage (4), Statistics(sizeInBytes=12.1 MiB, rowCount=1.00E+5)
                  :     +- Exchange (3)
                  :        +- * Project (2)
                  :           +- * Range (1)
                  +- BroadcastQueryStage (13), Statistics(sizeInBytes=8.0 MiB, rowCount=6)
                     +- BroadcastExchange (12)
                        +- AQEShuffleRead (11)
                           +- ShuffleQueryStage (10), Statistics(sizeInBytes=192.0 B, rowCount=6)
                              

In [44]:
csv_output_path = "/content/pyspark_sales_csv"
(sales_df.repartition(4)
.write
.mode("overwrite")
.option("header", True)
.csv(csv_output_path))
print("CSV output written to:", csv_output_path)
print("Part files:", [f for f in os.listdir(csv_output_path) if f.startswith("part-")])

CSV output written to: /content/pyspark_sales_csv
Part files: ['part-00000-416a80b9-4071-4fa5-94a6-358e139431cd-c000.csv', 'part-00001-416a80b9-4071-4fa5-94a6-358e139431cd-c000.csv', 'part-00003-416a80b9-4071-4fa5-94a6-358e139431cd-c000.csv', 'part-00002-416a80b9-4071-4fa5-94a6-358e139431cd-c000.csv']


**Final Challenge**

In [55]:
completed_df = sales_df.filter(F.col("order_status") == "Completed")

completed_df.show(10, truncate=False)

print("Total completed orders:", completed_df.count())

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|3       |2026-01-04|S02     |P04       |4       |17000     |0           |Completed   |UPI           |EMP04         |
|9       |2026-01-10|S04     |P04       |2       |17000     |5           |Completed   |UPI           |EMP10         |
|2       |2026-01-03|S01     |P03       |3       |3500      |0           |Completed   |Cash          |EMP03         |
|8       |2026-01-09|S03     |P03       |1       |3500      |5           |Completed   |Cash          |EMP09         |
|4       |2026-01-05|S02     |P05       |5       |4500      |0           |Completed   |Card          |EMP05         |
|10      |2026-01-11|S04     |P05       |3       |4500  

In [56]:
completed_df = completed_df.na.fill({"discount_pct": 0})

completed_df.select("order_id","discount_pct").show(10)

+--------+------------+
|order_id|discount_pct|
+--------+------------+
|       3|           0|
|       9|           5|
|       2|           0|
|       8|           5|
|       4|           0|
|      10|          10|
|       5|           5|
|       1|           0|
|      13|          10|
|       6|           5|
+--------+------------+
only showing top 10 rows


In [57]:
enriched_df = (completed_df
    .join(products_df, "product_id", "left")
    .join(stores_df, "store_id", "left"))
enriched_df.show(10, truncate=False)

enriched_df.printSchema()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+-----------+----------+---------+------+
|store_id|product_id|order_id|order_date|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|product_name|category   |list_price|city     |region|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+------------+-----------+----------+---------+------+
|S02     |P04       |3       |2026-01-04|4       |17000     |0           |Completed   |UPI           |EMP04         |Printer     |Office     |17000     |Bengaluru|South |
|S04     |P04       |9       |2026-01-10|2       |17000     |5           |Completed   |UPI           |EMP10         |Printer     |Office     |17000     |Delhi    |North |
|S01     |P03       |2       |2026-01-03|3       |3500      |0           |Completed   |Cash          |EMP03         |Headphones  |Accessories|350

In [59]:
revenue_df = (enriched_df
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("discounted_revenue",F.col("gross_revenue") * (1 - F.col("discount_pct") / F.lit(100.0))))


In [49]:
final_agg_df = (revenue_df
    .groupBy("region", "category")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"),
        F.sum("discounted_revenue").alias("total_discounted_revenue"),
        F.count("*").alias("transaction_count")))

In [51]:
region_window = Window.partitionBy("region").orderBy( F.desc("total_discounted_revenue"))
final_ranked_df = final_agg_df.withColumn(
    "category_revenue_rank",
    F.dense_rank().over(region_window))

In [52]:
final_cached_df = final_ranked_df.cache()
final_cached_df.count()

9

In [53]:
final_cached_df.orderBy("region", "category_revenue_rank").show(truncate=False)

+------+-----------+-------------------+------------------------+-----------------+---------------------+
|region|category   |total_gross_revenue|total_discounted_revenue|transaction_count|category_revenue_rank|
+------+-----------+-------------------+------------------------+-----------------+---------------------+
|North |Electronics|934848000          |8.551824E8              |6492             |1                    |
|North |Office     |441388000          |4.12114E8               |6492             |2                    |
|North |Networking |146097000          |1.35026775E8            |6494             |3                    |
|South |Electronics|2156236000         |2.0242794E9             |19484            |1                    |
|South |Office     |662524000          |6.060058E8              |6495             |2                    |
|South |Accessories|113631000          |1.05018025E8            |6494             |3                    |
|South |Networking |87669000           |8.2684

In [54]:
parquet_output_path = "/content/pyspark_final_revenue_parquet"
(final_cached_df
.repartition(2)
.write
.mode("overwrite")
.parquet(parquet_output_path))

print("Parquet output written to:", parquet_output_path)
print("Parquet part files:", [f for f in os.listdir(parquet_output_path) if f.startswith("part-")])

final_cached_df.unpersist()

Parquet output written to: /content/pyspark_final_revenue_parquet
Parquet part files: ['part-00001-168dea89-85d1-4369-b67a-8542a7967bf9-c000.snappy.parquet', 'part-00000-168dea89-85d1-4369-b67a-8542a7967bf9-c000.snappy.parquet']


DataFrame[region: string, category: string, total_gross_revenue: bigint, total_discounted_revenue: double, transaction_count: bigint, category_revenue_rank: int]